# 02 · Preprocessing the selected years (run on Colab)

For each year in `configs/data.json`:

1. download the archive from Zenodo to the Colab disk (15–18 GB, temporary),
2. read every sample **without extracting**, keep DBZ + VEL, lowest sweep, last frame, store as `uint8`,
3. save `tornet_<year>.npz` to Google Drive (~1–1.5 GB per year),
4. delete the archive.

**Run this on Colab with a CPU runtime** (no GPU needed; saves your GPU quota).
It takes a while (roughly 15–30 min per year, mostly the download). If the session disconnects, run it again:
finished years are skipped, and an unfinished download resumes.

Needs about 4–5 GB free on Google Drive.

In [ ]:
# --- Setup: works on Google Colab and on your own computer ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # processed data is kept on Drive
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        token = userdata.get("GITHUB_TOKEN")          # Colab Secrets (key icon on the left)
        !git clone -q https://{token}@github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Local", "| repo:", REPO)

In [ ]:
import shutil
import numpy as np
from src import paths, tornet

cfg = paths.load_config()
SCRATCH = paths.scratch_dir()      # temporary, large
OUT = paths.processed_dir()        # persistent (Drive on Colab)

def free_gb(p):
    return shutil.disk_usage(p).free / 1e9

print("years:", cfg["years"])
print(f"scratch: {SCRATCH}  ({free_gb(SCRATCH):.0f} GB free)")
print(f"output:  {OUT}  ({free_gb(OUT):.0f} GB free)")

In [ ]:
cat = tornet.get_catalog(paths.data_dir())

for year in cfg["years"]:
    out_file = OUT / f"tornet_{year}.npz"
    if out_file.exists():
        print(f"{year}: done already")
        continue
    tar_path = SCRATCH / f"tornet_{year}.tar.gz"
    print(f"{year}: downloading ...")
    tornet.download(tornet.tar_url(year), tar_path)
    print(f"{year}: processing ...")
    tornet.process_year(year, tar_path, OUT, cfg["variables"], cfg["sweep"], cfg["frame"], catalog=cat)
    tar_path.unlink()               # free the Colab disk for the next year
    print(f"{year}: archive deleted, {free_gb(SCRATCH):.0f} GB free on scratch")

## Check the results against the catalog

In [ ]:
for year in cfg["years"]:
    d = np.load(OUT / f"tornet_{year}.npz")
    X, y, names = d["X"], d["y"], d["filename"]
    expected = cat[cat["year"] == year]
    match = cat.set_index("filename").loc[names, "label"].to_numpy()
    print(f"{year}: X {X.shape} {X.dtype} | {len(y):,} samples (catalog {len(expected):,}) | "
          f"{int(y.sum()):,} tornado | labels match catalog: {(match == y).all()}")